# scVI best - human_pancreas

Seeds: `0-9`

Metrics: BAMI, CASW, SCGRAPHM, ISOLISI, BRAS, KBET, CPCR

This notebook is intentionally unexecuted.


In [1]:
import gc
import inspect
import json
import os
import sys
from pathlib import Path

os.environ["PYTHONNOUSERSITE"] = "1"
PROJECT_ROOT = (
    "/ubc/cs/research/beaver/projects/"
    "Single-cell-integration-metrics"
)
os.environ["R_HOME"] = (
    f"{PROJECT_ROOT}/CondaEnv/scvi_py310/lib/R"
)
os.environ["R_LIBS_SITE"] = (
    f"{PROJECT_ROOT}/CondaEnv/harmony_optuna/lib/R/library"
)
sys.path.insert(0, f"{PROJECT_ROOT}/scib")

OPTIMIZER_PATH = Path(
    PROJECT_ROOT,
    "scripts",
    "optuna_multi_metrics",
    "optuna_scvi",
    "optimize_scvi_multi_metrics.py",
)
sys.path.insert(0, str(OPTIMIZER_PATH.parent))

import anndata as ad
import numpy as np
import pandas as pd
import torch
import optimize_scvi_multi_metrics as scvi_workflow

assert Path(scvi_workflow.__file__).resolve() == OPTIMIZER_PATH


In [2]:
DATASET_NAME = 'human_pancreas'
DATA_PATH = f"{PROJECT_ROOT}/data/{DATASET_NAME}.h5ad"
BATCH_KEYS = ('tech',)
BATCH_KEY = BATCH_KEYS[0]
LABEL_KEY = 'celltype'

SELECTED_BIO_METRICS = ('BAMI', 'CASW', 'SCGRAPHM', 'ISOLISI')
SELECTED_BATCH_METRICS = ('BRAS', 'KBET', 'CPCR')
SELECTED_METRICS = SELECTED_BIO_METRICS + SELECTED_BATCH_METRICS

N_SEEDS = 10
N_NEIGHBORS = 15
USE_HVG = True
N_TOP_HVG = 2000
MAX_EPOCHS = 400

EMBED_KEY = "X_scvi"
CLUSTER_KEY = "leiden_opt"
METRIC_BATCH_KEY = "scvi_metric_batch"
METRIC_LABEL_KEY = "scvi_metric_label"

assert DATASET_NAME == 'human_pancreas'
assert BATCH_KEYS == ('tech',)
assert LABEL_KEY == 'celltype'
assert SELECTED_METRICS == ('BAMI', 'CASW', 'SCGRAPHM', 'ISOLISI', 'BRAS', 'KBET', 'CPCR')

pd.Series(
    {
        "dataset": DATASET_NAME,
        "data_path": DATA_PATH,
        "batch_keys": list(BATCH_KEYS),
        "label_key": LABEL_KEY,
        "metrics": list(SELECTED_METRICS),
        "seeds": list(range(N_SEEDS)),
        "use_hvg": USE_HVG,
        "n_top_hvg": N_TOP_HVG,
        "max_epochs": MAX_EPOCHS,
    },
    name="configuration",
)


dataset                                          human_pancreas
data_path     /ubc/cs/research/beaver/projects/Single-cell-i...
batch_keys                                               [tech]
label_key                                              celltype
metrics       [BAMI, CASW, SCGRAPHM, ISOLISI, BRAS, KBET, CPCR]
seeds                            [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
use_hvg                                                    True
n_top_hvg                                                  2000
max_epochs                                                  400
Name: configuration, dtype: object

In [3]:
scvi_workflow.DATA_PATH = DATA_PATH
scvi_workflow.DATASET_NAME = DATASET_NAME
scvi_workflow.BATCH_KEYS = list(BATCH_KEYS)
scvi_workflow.BATCH_KEY = BATCH_KEY
scvi_workflow.LABEL_KEY = LABEL_KEY
scvi_workflow.USE_HVG = USE_HVG
scvi_workflow.N_TOP_HVG = N_TOP_HVG
scvi_workflow.MAX_EPOCHS = MAX_EPOCHS
scvi_workflow.N_NEIGHBORS = N_NEIGHBORS
scvi_workflow.EMBED_KEY = EMBED_KEY
scvi_workflow.CLUSTER_KEY = CLUSTER_KEY
scvi_workflow.METRIC_BATCH_KEY = METRIC_BATCH_KEY
scvi_workflow.METRIC_LABEL_KEY = METRIC_LABEL_KEY

scvi_workflow.USE_BAMI = "BAMI" in SELECTED_METRICS
scvi_workflow.USE_CASW = "CASW" in SELECTED_METRICS
scvi_workflow.USE_SCGRAPHM = "SCGRAPHM" in SELECTED_METRICS
scvi_workflow.USE_TRAJ = "TRAJ" in SELECTED_METRICS
scvi_workflow.USE_ISOLISI = "ISOLISI" in SELECTED_METRICS
scvi_workflow.USE_BRAS = "BRAS" in SELECTED_METRICS
scvi_workflow.USE_KBET = "KBET" in SELECTED_METRICS
scvi_workflow.USE_CPCR = "CPCR" in SELECTED_METRICS

scvi_workflow.validate_configuration(SELECTED_METRICS)
scvi_workflow.validate_dataset_contents()

pd.Series(
    {
        "optimizer": str(OPTIMIZER_PATH),
        "scvi_version": scvi_workflow.scvi.__version__,
        "cuda_available": torch.cuda.is_available(),
    },
    name="preflight",
)


Dataset preflight: n_obs=16382 n_batches=9 n_labels=14 counts_layer=True
ISOLISI preflight: min_batch_coverage=4 isolated_labels=['t_cell']
CPCR preflight: eligible_labels=9


optimizer         /ubc/cs/research/beaver/projects/Single-cell-i...
scvi_version                                                  1.3.3
cuda_available                                                 True
Name: preflight, dtype: object

In [4]:
OPTUNA_BEST_SOURCE = '/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/results/optuna_multi_metrics/scvi/human_pancreas/expanded_metrics_v1/scvi_human_pancreas_joint_expanded_metrics_v1_best_params.json'
OPTUNA_TRIALS_SOURCE = '/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/results/optuna_multi_metrics/scvi/human_pancreas/expanded_metrics_v1/scvi_human_pancreas_joint_expanded_metrics_v1_trials.csv'
OPTUNA_STUDY_NAME = 'scvi_human_pancreas_joint_expanded_metrics_v1'
OPTUNA_BEST_TRIAL = 117
OPTUNA_BEST_VALUE = 0.7708637091353034
OPTUNA_BEST_PARAMS = {'n_latent': 31, 'n_hidden': 128, 'n_layers': 2, 'dropout_rate': 0.1545712897545604, 'lr': 0.0016045838047102994}

with open(OPTUNA_BEST_SOURCE, "r", encoding="utf-8") as source_file:
    current_source = json.load(source_file)
assert current_source["study_name"] == OPTUNA_STUDY_NAME
assert current_source["dataset_name"] == DATASET_NAME
assert current_source["best_value"] == OPTUNA_BEST_VALUE
assert current_source["best_params"] == OPTUNA_BEST_PARAMS
assert tuple(OPTUNA_BEST_PARAMS) == ('n_latent', 'n_hidden', 'n_layers', 'dropout_rate', 'lr')

SCVI_PARAMS = dict(OPTUNA_BEST_PARAMS)

pd.Series(
    {
        "source": OPTUNA_BEST_SOURCE,
        "trials_source": OPTUNA_TRIALS_SOURCE,
        "study": OPTUNA_STUDY_NAME,
        "best_trial": OPTUNA_BEST_TRIAL,
        "best_value": OPTUNA_BEST_VALUE,
        **SCVI_PARAMS,
        "gene_likelihood": "zinb",
        "dispersion": "gene",
        "latent_distribution": "normal",
        "early_stopping": True,
        "max_epochs": MAX_EPOCHS,
        "use_hvg": USE_HVG,
        "n_top_hvg": N_TOP_HVG,
    },
    name="best_setting",
)

def run_scvi_setting(adata, seed):
    return scvi_workflow.run_scvi(
        adata,
        n_latent=SCVI_PARAMS["n_latent"],
        n_hidden=SCVI_PARAMS["n_hidden"],
        n_layers=SCVI_PARAMS["n_layers"],
        dropout_rate=SCVI_PARAMS["dropout_rate"],
        lr=SCVI_PARAMS["lr"],
        seed=int(seed),
    )


In [5]:
def compute_joint_metrics(adata_raw, adata_int):
    result = scvi_workflow.compute_joint_metrics(adata_raw, adata_int)
    if tuple(result) != (
        *SELECTED_METRICS,
        "bio_score",
        "batch_score",
        "joint_score",
    ):
        raise ValueError(f"Unexpected metric result keys: {tuple(result)}")
    return result


In [6]:
results = []

for seed in range(N_SEEDS):
    print(f"\n===== seed {seed} =====", flush=True)

    adata_raw = ad.read_h5ad(DATA_PATH)
    adata_int = run_scvi_setting(adata_raw, seed=seed)
    metric_result = compute_joint_metrics(adata_raw, adata_int)

    row = {"seed": seed, **metric_result}
    results.append(row)
    print(pd.Series(row), flush=True)

    del adata_raw, adata_int
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

results_df = pd.DataFrame(results)
results_df



===== seed 0 =====


[rank: 0] Seed set to 0


GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


...

Cluster for leiden_opt_2.0 with leiden


resolution: 2.0, bAMI: 0.8628090816302008


optimised clustering against scvi_metric_label
optimal cluster resolution: 1.2
optimal score: 0.8934632770965806
BRAS...


seed           9.000000
BAMI           0.893463
CASW           0.574045
SCGRAPHM       0.798897
ISOLISI        0.965129
BRAS           0.765392
KBET           0.458309
CPCR           0.836803
bio_score      0.807883
batch_score    0.686835
joint_score    0.759464
dtype: float64


,seed,BAMI,CASW,SCGRAPHM,ISOLISI,BRAS,KBET,CPCR,bio_score,batch_score,joint_score
0,0,0.894645,0.585731,0.846398,0.971940,0.772369,0.465957,0.832099,0.824678,0.690142,0.770864
1,1,0.883785,0.575679,0.788216,0.898713,0.750839,0.447077,0.820769,0.786598,0.672895,0.741117
2,2,0.847051,0.587241,0.793513,0.957289,0.743159,0.432802,0.757876,0.796273,0.644612,0.735609
3,3,0.854629,0.579861,0.836708,0.945394,0.748904,0.457627,0.819369,0.804148,0.675300,0.752609
4,4,0.884128,0.587050,0.850149,0.967204,0.763880,0.448424,0.805134,0.822133,0.672479,0.762271
5,5,0.875887,0.582194,0.820998,0.858889,0.759412,0.459350,0.821834,0.784492,0.680199,0.742775
6,6,0.876255,0.578953,0.850737,0.902227,0.763342,0.438228,0.808338,0.802043,0.669969,0.749213
7,7,0.881317,0.578489,0.800225,0.834647,0.757969,0.440278,0.820913,0.773669,0.673053,0.733423
8,8,0.888628,0.590561,0.855396,0.891992,0.749417,0.430606,0.789945,0.806644,0.656656,0.746649
9,9,0.893463,0.574045,0.798897,0.965129,0.765392,0.458309,0.836803,0.807883,0.686835,0.759464


In [7]:
score_columns = [
    *SELECTED_METRICS,
    "bio_score",
    "batch_score",
    "joint_score",
]

summary_df = pd.DataFrame(
    {
        "mean": results_df[score_columns].mean(),
        "std": results_df[score_columns].std(ddof=0),
        "min": results_df[score_columns].min(),
        "max": results_df[score_columns].max(),
    }
)
summary_df


,mean,std,min,max
BAMI,0.877979,0.014909,0.847051,0.894645
CASW,0.581980,0.005193,0.574045,0.590561
SCGRAPHM,0.824124,0.025420,0.788216,0.855396
ISOLISI,0.919342,0.046440,0.834647,0.971940
BRAS,0.757468,0.008668,0.743159,0.772369
KBET,0.447866,0.011575,0.430606,0.465957
CPCR,0.811308,0.021904,0.757876,0.836803
bio_score,0.800856,0.015318,0.773669,0.824678
batch_score,0.672214,0.012722,0.644612,0.690142
joint_score,0.749399,0.011417,0.733423,0.770864
